# Modelo de targeting de SHM con SHazaM

SHazaM permite construir modelos de *targeting* de hipermutación somática (SHM) a partir de secuencias clonales previamente definidas mediante Change-O. Estos modelos estiman patrones de mutabilidad asociados al contexto nucleotídico de las mutaciones.

## Flujo del análisis

1. **Carga del repertorio**
   
   Se carga la tabla `clone-pass.tsv` con las secuencias clonales, alineamientos germinales e información génica necesaria para el análisis.

2. **Preparación y colapso clonal**
   
   Los clones definidos previamente mediante Change-O son colapsados utilizando `collapseClones()`, generando una secuencia consenso representativa para cada clon.

3. **Construcción del modelo de targeting**
   
   A partir de las secuencias consenso clonales se genera un modelo de targeting de SHM mediante `createTargetingModel()`, considerando los patrones de mutaciones silenciosas.

4. **Visualización del modelo**
   
   El patrón de mutabilidad obtenido se representa mediante `plotMutability()`, permitiendo evaluar la preferencia de mutación según el contexto de secuencia.

5. **Cálculo de distancia entre modelos**
   
   Finalmente, se calcula una matriz de distancia mediante `calcTargetingDistance()`, la cual permite comparar la similitud de los perfiles de targeting entre diferentes repertorios o escenarios simulados.

6. **Salida del análisis**
   
   El flujo genera como resultados:
   
   - Modelo de targeting de SHM.
   - Gráfico de mutabilidad.
   - Matriz de distancias entre modelos para análisis comparativo.

In [2]:
library(shazam)
library(alakazam)
library(readr)
library(dplyr)
library(ggplot2)

## Opción 1: Análisis individual de un repertorio

Se procesa un único archivo `clone-pass.tsv` como entrada para el análisis de targeting de SHM mediante SHazaM.

El flujo consiste en:

1. Cargar y preparar el archivo, eliminando secuencias sin clon asignado o sin información germinal válida (`NA` o campos vacíos).
2. Colapsar secuencias clonales mediante `collapseClones()`.
3. Construir el modelo de targeting de SHM con `createTargetingModel()`.
4. Guardar el objeto `TargetingModel` en formato `.rds` para su reutilización.
5. Generar el gráfico de mutabilidad (`plotMutability`) como salida visual.

**Entradas:**
- Un archivo `clone-pass.tsv`.

**Salidas:**
- Un objeto `TargetingModel` (`.rds`).
- Un gráfico de mutabilidad (`.png`).

In [ ]:


# 1. Cargar archivo clone-pass
archivo_clones <- "../data/output/repertorio_D_insilico_6400_seqs_clone-pass.tsv"

db <- read_tsv(
  archivo_clones,
  show_col_types = FALSE
) %>%
  mutate(
    sample_id = "repertorio_D",
    clone_id = as.character(clone_id)
  ) %>%
  filter(!is.na(clone_id))


# 2. Asignar IGHM si c_call está completamente vacío
if (all(is.na(db$c_call))) {
  db$c_call <- "IGHM"
}


# 3. Filtrar secuencias con germline válida
db_shazam <- db %>%
  filter(
    !is.na(germline_alignment),
    germline_alignment != ""
  )


# 4. Revisar datos
cat("Secuencias originales:", nrow(db), "\n")
cat("Secuencias para SHazaM:", nrow(db_shazam), "\n")
cat("NA germline:", sum(is.na(db_shazam$germline_alignment)), "\n")
cat("Vacías germline:", sum(db_shazam$germline_alignment == ""), "\n")


# 5. Colapsar clones
clones <- collapseClones(
  db_shazam,
  cloneColumn = "clone_id",
  sequenceColumn = "sequence_alignment",
  germlineColumn = "germline_alignment",
  regionDefinition = IMGT_V,
  method = "thresholdedFreq",
  minimumFrequency = 0.6,
  nproc = 1
)


# 6. Crear modelo de targeting SHM
targeting_model <- createTargetingModel(
  clones,
  model = "s",
  sequenceColumn = "clonal_sequence",
  germlineColumn = "clonal_germline",
  vCallColumn = "v_call"
)


# 7. Guardar modelo y gráfico si existe
if (is.null(targeting_model) || length(targeting_model) == 0) {

  message(
    "No se generó modelo de targeting: no hay suficientes mutaciones."
  )

} else {

  message(
    "Modelo de targeting generado correctamente."
  )


  # Guardar objeto TargetingModel
  saveRDS(
    targeting_model,
    "../results/shm_models/targeting/repertorio_D_6400_targeting_model.rds"
  )


  # Guardar gráfico hedgehog para C
  png(
    "../results/shm_models/targeting/repertorio_D_6400seq_targeting_C.png",
    width = 2000,
    height = 1500,
    res = 300
  )

  plotMutability(
    targeting_model,
    nucleotides = "C",
    style = "hedgehog"
  )

  dev.off()


  message(
    "Modelo y gráfico guardados correctamente."
  )

}

## Opción 2: Análisis de múltiples profundidades

Se procesaron múltiples archivos `clone-pass.tsv` correspondientes a diferentes profundidades de secuenciación del mismo escenario experimental.

Para cada profundidad se realizó el mismo flujo de análisis:

1. Carga del archivo `clone-pass.tsv`.
2. Filtrado de clonotipos sin asignación y secuencias sin información germinal válida (`NA` o campos vacíos).
3. Colapso de secuencias clonales mediante `collapseClones()`.
4. Inferencia del modelo de targeting de SHM mediante `createTargetingModel()`.
5. Generación del perfil de mutabilidad con `plotMutability()`.

Los modelos generados para cada profundidad fueron almacenados en una lista y guardados como un único objeto `.rds`, permitiendo recuperar posteriormente cualquier modelo individual sin repetir el procesamiento.

**Entrada:**
- Múltiples archivos `clone-pass.tsv` correspondientes a profundidades de 100 a 102400 secuencias.

**Salidas:**
- Un gráfico de mutabilidad (`.png`) por profundidad.
- Un archivo `.rds` que contiene todos los objetos `TargetingModel` del escenario analizado.

In [ ]:


# Archivos clone-pass
ruta <- "/Users/catg/Desktop/SOFIAC/Gitsofia/tesisbioinf-sofia/data/output/"


# Salidas (carpetas ya creadas)
ruta_plots <- "/Users/catg/Desktop/SOFIAC/Gitsofia/tesisbioinf-sofia/results/shm_models/targeting/plots/"

ruta_models <- "/Users/catg/Desktop/SOFIAC/Gitsofia/tesisbioinf-sofia/results/shm_models/targeting/models/"

# Profundidades

depths <- c(
  100, 200, 400, 800,
  1600, 3200, 6400,
  12800, 25600, 51200,
  102400
)

# Lista para guardar modelos

targeting_models <- list()


# Procesamiento por profundidad

for (n in depths) {
  
  cat("\nProcesando escenario E -", n, "secuencias\n")
    
  # Archivo de entrada
  
  archivo <- paste0(
    ruta,
    "repertorio_E_insilico_",
    n,
    "_seqs_clone-pass.tsv"
  )
  
  # Verificar archivo
  
  if (!file.exists(archivo)) {
    
    cat("No existe:", archivo, "\n")
    next
    
  }
  
  # Cargar datos
  
 db <- read_tsv(
    archivo,
    show_col_types = FALSE
  ) %>%
    mutate(
      sample_id = paste0("E_", n),
      clone_id = as.character(clone_id)
    ) %>%
    filter(!is.na(clone_id))

  # Asignar región constante
 
  
  if(all(is.na(db$c_call))){
    
    db$c_call <- "IGHM"
    
  }
  
  # Filtrar germline faltante

  
  db_shazam <- db %>%
    filter(
      !is.na(germline_alignment),
      germline_alignment != ""
    )
  
  
  cat(
    "Secuencias usadas:",
    nrow(db_shazam),
    "\n"
  )

  # Collapse clones
  
  
  clones <- collapseClones(
    db_shazam,
    cloneColumn = "clone_id",
    sequenceColumn = "sequence_alignment",
    germlineColumn = "germline_alignment",
    regionDefinition = IMGT_V,
    method = "thresholdedFreq",
    minimumFrequency = 0.6,
    nproc = 1
  )

  # Crear modelo targeting
  targeting_model <- createTargetingModel(
    clones,
    model = "s",
    sequenceColumn = "clonal_sequence",
    germlineColumn = "clonal_germline",
    vCallColumn = "v_call"
  )

  # Guardar modelo en lista

  nombre <- paste0("E_", n)
  
  targeting_models[[nombre]] <- targeting_model

  # Guardar gráfico

  png(
    filename = paste0(
      ruta_plots,
      nombre,
      "_targeting_C.png"
    ),
    width = 2000,
    height = 1500,
    res = 300
  )
  
  plotMutability(
    targeting_model,
    nucleotides = "C",
    style = "hedgehog"
  )
  
 dev.off()
  
  cat("Terminado:", nombre, "\n")
  
}
# Guardar modelos completos del escenario E


saveRDS(
  targeting_models,
  paste0(
    ruta_models,
    "E_all_targeting_models.rds"
  )
)


cat("\nEscenario E terminado correctamente\n")


Procesando escenario E - 100 secuencias
Secuencias usadas: 100 


Warning message in createMutabilityMatrix(db, sub_mat, model = model, sequenceColumn = sequenceColumn, :
"Insufficient number of mutations to infer some 5-mers. Filled with 0. "


Terminado: E_100 

Procesando escenario E - 200 secuencias
Secuencias usadas: 199 


Warning message in createMutabilityMatrix(db, sub_mat, model = model, sequenceColumn = sequenceColumn, :
"Insufficient number of mutations to infer some 5-mers. Filled with 0. "


Terminado: E_200 

Procesando escenario E - 400 secuencias
Secuencias usadas: 400 


...

Terminado: E_25600 

Procesando escenario E - 51200 secuencias
Secuencias usadas: 51177 


Warning message in createMutabilityMatrix(db, sub_mat, model = model, sequenceColumn = sequenceColumn, :
"Insufficient number of mutations to infer some 5-mers. Filled with 0. "


Terminado: E_51200 

Procesando escenario E - 102400 secuencias
Secuencias usadas: 102355 


Warning message in createMutabilityMatrix(db, sub_mat, model = model, sequenceColumn = sequenceColumn, :
"Insufficient number of mutations to infer some 5-mers. Filled with 0. "


Terminado: E_102400 

Escenario E terminado correctamente


## Cálculo de matriz de distancia de targeting (paso adicional)

A partir del objeto `TargetingModel` generado por SHazaM se calculó una matriz de distancia de targeting mediante la función `calcTargetingDistance()`.

Este paso transforma las probabilidades de mutabilidad aprendidas por el modelo en una matriz de distancias entre motivos de 5 nucleótidos (5-mers), la cual puede ser utilizada posteriormente en análisis de agrupamiento clonal basados en modelos de targeting.

**Entrada:**
- Objeto `TargetingModel` almacenado en formato `.rds`.

**Proceso:**
1. Cargar el modelo de targeting correspondiente a una profundidad específica.
2. Aplicar `calcTargetingDistance()` para generar la matriz de distancia.
3. Exportar la matriz en formato `.tsv`.

**Salida:**
- Matriz de distancia de targeting (`.tsv`).

In [ ]:

# Cargar modelos guardados
E_models <- readRDS(
  "../results/shm_models/targeting/models/E_all_targeting_models.rds"
)

# Elegir profundidad
modelo_E_102400 <- E_models$E_102400

# Calcular matriz de distancia
targeting_dist <- calcTargetingDistance(modelo_E_102400)

# Guardar matriz como TSV
write_tsv(
  as.data.frame(targeting_dist),
  "../results/shm_models/targeting/E_102400_targeting_distance.tsv"
)
